In [1]:
!git clone https://github.com/Team-TUD/CTAB-GAN-Plus
import sys
sys.path.append('./CTAB-GAN-Plus')
from model.ctabgan import CTABGAN

fatal: destination path 'CTAB-GAN-Plus' already exists and is not an empty directory.


In [2]:
pip install sdv

In [3]:
pip install ucimlrepo

In [4]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

from sdv.metadata import SingleTableMetadata
from sdv.single_table import (
    CTGANSynthesizer,
    CopulaGANSynthesizer,
    TVAESynthesizer,
    GaussianCopulaSynthesizer
)

from sdv.evaluation.single_table import evaluate_quality

from model.ctabgan import CTABGAN

bank_marketing = fetch_ucirepo(id=222)

X = bank_marketing.data.features
y = bank_marketing.data.targets

data = pd.concat([X, y], axis=1)

target_col = y.columns[0]

data = data.replace("unknown", np.nan)

data[target_col] = data[target_col].astype(str).str.strip()
data[target_col] = data[target_col].map({"yes": 1, "no": 0})

date_cols = ["day_of_week", "month", "duration"]
data = data.drop(columns=[col for col in date_cols if col in data.columns])

data = data.dropna(subset=[target_col])

n_samples = min(10000, len(data))
data = data.sample(n=n_samples, random_state=42).reset_index(drop=True)

numeric_cols = data.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = data.select_dtypes(include=["object", "category"]).columns.tolist()

if target_col in numeric_cols:
    numeric_cols.remove(target_col)

for col in numeric_cols:
    data[col] = data[col].fillna(data[col].mean())

for col in categorical_cols:
    data[col] = data[col].fillna(data[col].mode()[0])

label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))
    label_encoders[col] = le

data[target_col] = data[target_col].astype(int)

X = data.drop(columns=[target_col])
y = data[target_col]

processed_data = pd.concat([X, y], axis=1)

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(processed_data)

metadata.update_column(
    column_name=target_col,
    sdtype="categorical"
)

N_SAMPLES = 10000
TEST_SIZE = 0.2
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

scores = {}
synthetic_datasets = {}
quality_results = []


In [5]:
seed = SEED

print("\n================ SINGLE RUN ================")

np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

train_real, test_real = train_test_split(
    processed_data,
    test_size=TEST_SIZE,
    stratify=processed_data[target_col],
    random_state=seed
)

train_metadata = SingleTableMetadata()
train_metadata.detect_from_dataframe(train_real)

train_metadata.update_column(
    column_name=target_col,
    sdtype="categorical"
)

try:
    data_path = "bank_marketing_train.csv"
    train_real.to_csv(data_path, index=False)

    categorical_columns = [
        col for col in train_real.columns
        if col != target_col and col in label_encoders
    ]

    ctabgan = CTABGAN(
        raw_csv_path=data_path,
        categorical_columns=categorical_columns + [target_col],
        log_columns=[],
        mixed_columns={},
        integer_columns=[],
        problem_type={"Classification": target_col}
    )

    ctabgan.fit()

    synthetic_ctabgan = ctabgan.data_prep.inverse_prep(
        ctabgan.synthesizer.sample(N_SAMPLES)
    )

    if target_col in synthetic_ctabgan.columns:
        synthetic_ctabgan[target_col] = pd.to_numeric(
            synthetic_ctabgan[target_col],
            errors="coerce"
        )

        synthetic_ctabgan[target_col] = (
            synthetic_ctabgan[target_col]
            .fillna(train_real[target_col].mode()[0])
            .round()
            .astype(int)
            .clip(0, 1)
        )

        if synthetic_ctabgan[target_col].nunique() < 2:
            real_dist = train_real[target_col].value_counts(normalize=True)

            synthetic_ctabgan[target_col] = np.random.choice(
                real_dist.index,
                size=len(synthetic_ctabgan),
                p=real_dist.values
            )

    synthetic_datasets["CTABGAN"] = synthetic_ctabgan.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_ctabgan,
        metadata=train_metadata
    )

    score = quality.get_score()

    scores["CTABGAN"] = score

    print(
        "CTABGAN:",
        round(score, 4),
        "| Target Distribution:",
        synthetic_ctabgan[target_col].value_counts(normalize=True).to_dict()
    )

except Exception as e:
    print("CTABGAN Failed:", e)


================ SINGLE RUN ================


100%|██████████| 150/150 [10:53<00:00,  4.36s/it]


Finished training in 657.9997153282166  seconds.
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 14/14 [00:01<00:00,  7.76it/s]|
Column Shapes Score: 29.44%

(2/2) Evaluating Column Pair Trends: |██████████| 91/91 [00:00<00:00, 260.69it/s]|
Column Pair Trends Score: 24.76%

Overall Score (Average): 27.1%

CTABGAN: 0.271 | Target Distribution: {0: 0.8608, 1: 0.1392}


In [6]:
# WGAN-GP

try:

    import traceback

    data_wgan = train_real.copy()

    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(data_wgan)

    device = "cuda" if torch.cuda.is_available() else "cpu"

    real_tensor = torch.tensor(
        scaled_data,
        dtype=torch.float32
    )

    batch_size = 64
    latent_dim = 64
    data_dim = real_tensor.shape[1]

    loader = torch.utils.data.DataLoader(
        real_tensor,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False
    )

    class Generator(nn.Module):
        def __init__(self):
            super().__init__()

            self.model = nn.Sequential(
                nn.Linear(latent_dim, 128),
                nn.LayerNorm(128),
                nn.LeakyReLU(0.2),

                nn.Linear(128, 256),
                nn.LayerNorm(256),
                nn.LeakyReLU(0.2),

                nn.Linear(256, data_dim)
            )

        def forward(self, z):
            return self.model(z)

    class Critic(nn.Module):
        def __init__(self):
            super().__init__()

            self.model = nn.Sequential(
                nn.Linear(data_dim, 256),
                nn.LeakyReLU(0.2),

                nn.Linear(256, 128),
                nn.LeakyReLU(0.2),

                nn.Linear(128, 1)
            )

        def forward(self, x):
            return self.model(x)

    generator = Generator().to(device)
    critic = Critic().to(device)

    optimizer_G = optim.Adam(
        generator.parameters(),
        lr=0.0001,
        betas=(0.5, 0.9)
    )

    optimizer_C = optim.Adam(
        critic.parameters(),
        lr=0.0001,
        betas=(0.5, 0.9)
    )

    def gradient_penalty(critic, real_samples, fake_samples):

        alpha = torch.rand(real_samples.size(0), 1, device=device)
        alpha = alpha.expand_as(real_samples)

        interpolates = (
            alpha * real_samples +
            (1 - alpha) * fake_samples
        ).requires_grad_(True)

        critic_interpolates = critic(interpolates)

        gradients = torch.autograd.grad(
            outputs=critic_interpolates,
            inputs=interpolates,
            grad_outputs=torch.ones_like(critic_interpolates),
            create_graph=True,
            retain_graph=True
        )[0]

        gradients = gradients.view(gradients.size(0), -1)

        return ((gradients.norm(2, dim=1) - 1) ** 2).mean()

    for epoch in range(100):

        for real_batch in loader:

            real_batch = real_batch.to(device)

            for _ in range(5):

                z = torch.randn(
                    real_batch.size(0),
                    latent_dim,
                    device=device
                )

                fake_batch = generator(z).detach()

                critic_real = critic(real_batch).mean()
                critic_fake = critic(fake_batch).mean()

                gp = gradient_penalty(
                    critic,
                    real_batch,
                    fake_batch
                )

                critic_loss = (
                    critic_fake
                    - critic_real
                    + 10 * gp
                )

                optimizer_C.zero_grad()
                critic_loss.backward()
                optimizer_C.step()

            z = torch.randn(
                real_batch.size(0),
                latent_dim,
                device=device
            )

            fake = generator(z)

            generator_loss = -critic(fake).mean()

            optimizer_G.zero_grad()
            generator_loss.backward()
            optimizer_G.step()

    generator.eval()

    with torch.no_grad():

        z = torch.randn(
            N_SAMPLES,
            latent_dim,
            device=device
        )

        synthetic_scaled = generator(z).cpu().numpy()

    synthetic = scaler.inverse_transform(synthetic_scaled)

    synthetic_wgan = pd.DataFrame(
        synthetic,
        columns=data_wgan.columns
    )

    for col in synthetic_wgan.columns:

        if col == target_col:
            continue

        if col in label_encoders:
            max_val = processed_data[col].max()
            min_val = processed_data[col].min()

            synthetic_wgan[col] = (
                synthetic_wgan[col]
                .round()
                .clip(min_val, max_val)
                .astype(int)
            )

    real_target_dist = train_real[target_col].value_counts(normalize=True)

    synthetic_wgan[target_col] = np.random.choice(
        real_target_dist.index,
        size=len(synthetic_wgan),
        p=real_target_dist.values
    )

    synthetic_wgan[target_col] = synthetic_wgan[target_col].astype(int)

    synthetic_datasets["WGAN_GP"] = synthetic_wgan.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_wgan,
        metadata=train_metadata
    )

    scores["WGAN_GP"] = quality.get_score()

    print(
        "WGAN_GP:",
        round(scores["WGAN_GP"], 4),
        "| Target Distribution:",
        synthetic_wgan[target_col].value_counts(normalize=True).to_dict()
    )

    del generator
    del critic
    del real_tensor

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

except Exception as e:

    print("WGAN_GP Failed:")
    traceback.print_exc()

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 14/14 [00:02<00:00,  5.81it/s]|
Column Shapes Score: 87.89%

(2/2) Evaluating Column Pair Trends: |██████████| 91/91 [00:00<00:00, 283.02it/s]|
Column Pair Trends Score: 72.22%

Overall Score (Average): 80.06%

WGAN_GP: 0.8006 | Target Distribution: {0: 0.8725, 1: 0.1275}


In [7]:
# SDV MODELS

sdv_models = {
    "CTGAN": CTGANSynthesizer(metadata=train_metadata),
    "CopulaGAN": CopulaGANSynthesizer(metadata=train_metadata),
    "TVAE": TVAESynthesizer(metadata=train_metadata),
    "GaussianCopula": GaussianCopulaSynthesizer(metadata=train_metadata)
}

for model_name, model in sdv_models.items():

    try:

        model.fit(train_real)

        synthetic_data = model.sample(N_SAMPLES)

        synthetic_datasets[model_name] = synthetic_data.copy()

        quality = evaluate_quality(
            real_data=train_real,
            synthetic_data=synthetic_data,
            metadata=train_metadata
        )

        scores[model_name] = quality.get_score()

        print(
            f"{model_name}: {round(scores[model_name], 4)}"
        )

    except Exception as e:

        print(
            f"{model_name} Failed: {e}"
        )

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 14/14 [00:00<00:00, 19.11it/s]|
Column Shapes Score: 92.9%

(2/2) Evaluating Column Pair Trends: |██████████| 91/91 [00:00<00:00, 285.05it/s]|
Column Pair Trends Score: 88.37%

Overall Score (Average): 90.63%

CTGAN: 0.9063
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 14/14 [00:01<00:00, 13.84it/s]|
Column Shapes Score: 90.13%

(2/2) Evaluating Column Pair Trends: |██████████| 91/91 [00:00<00:00, 252.87it/s]|
Column Pair Trends Score: 87.36%

Overall Score (Average): 88.74%

CopulaGAN: 0.8874
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 14/14 [00:01<00:00,  9.72it/s]|
Column Shapes Score: 90.72%

(2/2) Evaluating Column Pair Trends: |██████████| 91/91 [00:00<00:00, 276.08it/s]|
Column Pair Trends Score: 84.0%

Overall Score (Average): 87.36%

TVAE: 0.8736
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 14/14 [00:01<00:00,  9.27it/s]|
Column Shapes Score:

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier


models = {

    'LogReg': LogisticRegression(max_iter=5000, solver='liblinear', random_state=42),
    'SVM-RBF': SVC(kernel='rbf', probability=True, random_state=42),
    'KNN': KNeighborsClassifier(),
    'NaiveBayes': GaussianNB(),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(random_state=42),
    'ExtraTrees':  ExtraTreesClassifier(random_state=42),
    'GradientBoost': GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "MLP": MLPClassifier(max_iter=2000, random_state=42),
}

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.base import clone
import pandas as pd
import numpy as np

def evaluate_models(
    train_df,
    test_df,
    label_col,
    models,
    test_size=0.2,
    seeds=None
):
    if seeds is None:
        seeds = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

    results = []

    for name, model in models.items():

        accuracy_scores = []
        f1_scores = []
        precision_scores = []
        recall_scores = []

        for seed in seeds:

            X_train_full = train_df.drop(columns=[label_col])
            y_train_full = train_df[label_col]

            X_test_full = test_df.drop(columns=[label_col])
            y_test_full = test_df[label_col]

            stratify_train = (
                y_train_full
                if y_train_full.nunique() > 1 and y_train_full.value_counts().min() >= 2
                else None
            )

            X_train_split, _, y_train_split, _ = train_test_split(
                X_train_full,
                y_train_full,
                test_size=test_size,
                random_state=seed,
                stratify=stratify_train
            )

            stratify_test = (
                y_test_full
                if y_test_full.nunique() > 1 and y_test_full.value_counts().min() >= 2
                else None
            )

            _, X_test_split, _, y_test_split = train_test_split(
                X_test_full,
                y_test_full,
                test_size=test_size,
                random_state=seed,
                stratify=stratify_test
            )

            if y_train_split.nunique() < 2:
                continue

            scaler = StandardScaler()

            X_train_s = scaler.fit_transform(X_train_split)
            X_test_s = scaler.transform(X_test_split)

            clf = clone(model)

            if hasattr(clf, "random_state"):
                clf.set_params(random_state=seed)

            try:
                clf.fit(X_train_s, y_train_split)

                y_pred = clf.predict(X_test_s)

                accuracy_scores.append(
                    accuracy_score(y_test_split, y_pred)
                )

                f1_scores.append(
                    f1_score(
                        y_test_split,
                        y_pred,
                        average="weighted",
                        zero_division=0
                    )
                )

                precision_scores.append(
                    precision_score(
                        y_test_split,
                        y_pred,
                        average="weighted",
                        zero_division=0
                    )
                )

                recall_scores.append(
                    recall_score(
                        y_test_split,
                        y_pred,
                        average="weighted",
                        zero_division=0
                    )
                )

            except Exception:
                continue

        if len(accuracy_scores) == 0:
            results.append({
                "Model": name,
                "Accuracy Mean": np.nan,
                "Accuracy Std": np.nan,
                "F1 Mean": np.nan,
                "F1 Std": np.nan,
                "Precision Mean": np.nan,
                "Precision Std": np.nan,
                "Recall Mean": np.nan,
                "Recall Std": np.nan,
                "Accuracy ± SD": "N/A",
                "F1 ± SD": "N/A",
                "Precision ± SD": "N/A",
                "Recall ± SD": "N/A",
                "Accuracy (Mean±Std)": "N/A",
                "F1 (Mean±Std)": "N/A",
                "Precision (Mean±Std)": "N/A",
                "Recall (Mean±Std)": "N/A"
            })
            continue

        acc_mean = np.mean(accuracy_scores)
        acc_std = np.std(accuracy_scores, ddof=1) if len(accuracy_scores) > 1 else 0

        f1_mean = np.mean(f1_scores)
        f1_std = np.std(f1_scores, ddof=1) if len(f1_scores) > 1 else 0

        prec_mean = np.mean(precision_scores)
        prec_std = np.std(precision_scores, ddof=1) if len(precision_scores) > 1 else 0

        rec_mean = np.mean(recall_scores)
        rec_std = np.std(recall_scores, ddof=1) if len(recall_scores) > 1 else 0

        results.append({
            "Model": name,

            "Accuracy Mean": acc_mean,
            "Accuracy Std": acc_std,
            "F1 Mean": f1_mean,
            "F1 Std": f1_std,
            "Precision Mean": prec_mean,
            "Precision Std": prec_std,
            "Recall Mean": rec_mean,
            "Recall Std": rec_std,

            "Accuracy ± SD": f"{acc_mean:.4f} ± {acc_std:.4f}",
            "F1 ± SD": f"{f1_mean:.4f} ± {f1_std:.4f}",
            "Precision ± SD": f"{prec_mean:.4f} ± {prec_std:.4f}",
            "Recall ± SD": f"{rec_mean:.4f} ± {rec_std:.4f}",

            "Accuracy (Mean±Std)": f"{acc_mean:.4f} ± {acc_std:.4f}",
            "F1 (Mean±Std)": f"{f1_mean:.4f} ± {f1_std:.4f}",
            "Precision (Mean±Std)": f"{prec_mean:.4f} ± {prec_std:.4f}",
            "Recall (Mean±Std)": f"{rec_mean:.4f} ± {rec_std:.4f}"
        })

    return pd.DataFrame(results).sort_values(
        by="Accuracy Mean",
        ascending=False,
        na_position="last"
    )

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.base import clone
import pandas as pd
import numpy as np

print("--- Starting TRTR Evaluation (Train Real, Test Real) ---")

trtr_results = []

SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

X = processed_data.drop(columns=[target_col])
y = processed_data[target_col]

print("Target column:", target_col)
print("Target classes:")
print(y.value_counts())

for model_name, model in models.items():

    accuracy_scores = []
    f1_scores = []
    precision_scores = []
    recall_scores = []

    print(f"Running {model_name}...")

    for seed in SEEDS:

        X_train_real, X_test_real, y_train_real, y_test_real = train_test_split(
            X,
            y,
            test_size=TEST_SIZE,
            stratify=y,
            random_state=seed
        )

        scaler = StandardScaler()

        X_train_real = scaler.fit_transform(X_train_real)
        X_test_real = scaler.transform(X_test_real)

        clf = clone(model)

        if hasattr(clf, "random_state"):
            clf.set_params(random_state=seed)

        clf.fit(X_train_real, y_train_real)

        y_pred = clf.predict(X_test_real)

        accuracy_scores.append(
            accuracy_score(y_test_real, y_pred)
        )

        f1_scores.append(
            f1_score(
                y_test_real,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

        precision_scores.append(
            precision_score(
                y_test_real,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

        recall_scores.append(
            recall_score(
                y_test_real,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

    acc_mean = np.mean(accuracy_scores)
    acc_std = np.std(accuracy_scores, ddof=1)

    f1_mean = np.mean(f1_scores)
    f1_std = np.std(f1_scores, ddof=1)

    prec_mean = np.mean(precision_scores)
    prec_std = np.std(precision_scores, ddof=1)

    rec_mean = np.mean(recall_scores)
    rec_std = np.std(recall_scores, ddof=1)

    trtr_results.append({
        "Model": model_name,
        "Accuracy Mean_TRTR": acc_mean,
        "Accuracy Std_TRTR": acc_std,
        "F1 Mean_TRTR": f1_mean,
        "F1 Std_TRTR": f1_std,
        "Precision Mean_TRTR": prec_mean,
        "Precision Std_TRTR": prec_std,
        "Recall Mean_TRTR": rec_mean,
        "Recall Std_TRTR": rec_std,
        "Accuracy (Mean±Std)_TRTR": f"{acc_mean:.4f} ± {acc_std:.4f}",
        "F1 (Mean±Std)_TRTR": f"{f1_mean:.4f} ± {f1_std:.4f}",
        "Precision (Mean±Std)_TRTR": f"{prec_mean:.4f} ± {prec_std:.4f}",
        "Recall (Mean±Std)_TRTR": f"{rec_mean:.4f} ± {rec_std:.4f}"
    })

trtr_results_df = pd.DataFrame(trtr_results).sort_values(
    by="Accuracy Mean_TRTR",
    ascending=False
)

display(
    trtr_results_df[
        [
            "Model",
            "Accuracy (Mean±Std)_TRTR",
            "F1 (Mean±Std)_TRTR",
            "Precision (Mean±Std)_TRTR",
            "Recall (Mean±Std)_TRTR"
        ]
    ]
)

--- Starting TRTR Evaluation (Train Real, Test Real) ---
Target column: y
Target classes:
y
0    8794
1    1206
Name: count, dtype: int64
Running LogReg...
Running SVM-RBF...
Running KNN...
Running NaiveBayes...
Running DecisionTree...
Running RandomForest...
Running ExtraTrees...
Running GradientBoost...
Running AdaBoost...
Running MLP...


,Model,Accuracy (Mean±Std)_TRTR,F1 (Mean±Std)_TRTR,Precision (Mean±Std)_TRTR,Recall (Mean±Std)_TRTR
1,SVM-RBF,0.8890 ± 0.0030,0.8577 ± 0.0043,0.8680 ± 0.0073,0.8890 ± 0.0030
0,LogReg,0.8885 ± 0.0028,0.8570 ± 0.0045,0.8666 ± 0.0070,0.8885 ± 0.0028
7,GradientBoost,0.8884 ± 0.0036,0.8594 ± 0.0045,0.8657 ± 0.0082,0.8884 ± 0.0036
8,AdaBoost,0.8883 ± 0.0041,0.8584 ± 0.0056,0.8656 ± 0.0097,0.8883 ± 0.0041
9,MLP,0.8848 ± 0.0055,0.8586 ± 0.0069,0.8587 ± 0.0107,0.8848 ± 0.0055
5,RandomForest,0.8831 ± 0.0058,0.8575 ± 0.0066,0.8557 ± 0.0106,0.8831 ± 0.0058
2,KNN,0.8797 ± 0.0028,0.8544 ± 0.0038,0.8496 ± 0.0051,0.8797 ± 0.0028
6,ExtraTrees,0.8702 ± 0.0046,0.8487 ± 0.0047,0.8392 ± 0.0065,0.8702 ± 0.0046
3,NaiveBayes,0.8510 ± 0.0078,0.8419 ± 0.0068,0.8348 ± 0.0068,0.8510 ± 0.0078
4,DecisionTree,0.8092 ± 0.0098,0.8142 ± 0.0068,0.8197 ± 0.0046,0.8092 ± 0.0098


In [11]:
import pandas as pd
import numpy as np

label_col = target_col

model_order = [
    "CTGAN",
    "CopulaGAN",
    "TVAE",
    "GaussianCopula",
    "WGAN_GP",
    "CTABGAN"
]

seeds = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

real_data = processed_data.copy()

print("TRTR (Train Real, Test Real)")

trtr_results = evaluate_models(
    train_df=real_data,
    test_df=real_data,
    label="y",
    models=models,
    test_size=TEST_SIZE,
    seeds=seeds
)

display(
    trtr_results[
        [
            "Model",
            "Accuracy ± SD",
            "F1 ± SD",
            "Precision ± SD",
            "Recall ± SD"
        ]
    ]
)

print("=" * 70)

all_comparisons = []

for synth_name in model_order:

    if synth_name not in synthetic_datasets:
        print(f"{synth_name} not found in synthetic_datasets. Skipping.")
        continue

    print(f"{synth_name} - TSTR")

    synthetic_train_df = synthetic_datasets[synth_name].copy()

    if label_col not in synthetic_train_df.columns:
        print(f"{label_col} not found in {synth_name}. Skipping.")
        continue

    synthetic_train_df[label_col] = pd.to_numeric(
        synthetic_train_df[label_col],
        errors="coerce"
    )

    synthetic_train_df[label_col] = (
        synthetic_train_df[label_col]
        .fillna(real_data[label_col].mode()[0])
        .round()
        .astype(int)
    )

    synthetic_train_df = synthetic_train_df.dropna()

    tstr_results = evaluate_models(
        train_df=synthetic_train_df,
        test_df=real_data,
        label="y",
        models=models,
        test_size=TEST_SIZE,
        seeds=seeds
    )

    display(
        tstr_results[
            [
                "Model",
                "Accuracy ± SD",
                "F1 ± SD",
                "Precision ± SD",
                "Recall ± SD"
            ]
        ]
    )

    comparison = trtr_results.merge(
        tstr_results,
        on="Model",
        suffixes=("_TRTR", "_TSTR")
    )

    comparison["Accuracy_Drop"] = (
        comparison["Accuracy Mean_TRTR"]
        - comparison["Accuracy Mean_TSTR"]
    )

    comparison["F1_Drop"] = (
        comparison["F1 Mean_TRTR"]
        - comparison["F1 Mean_TSTR"]
    )

    comparison["Precision_Drop"] = (
        comparison["Precision Mean_TRTR"]
        - comparison["Precision Mean_TSTR"]
    )

    comparison["Recall_Drop"] = (
        comparison["Recall Mean_TRTR"]
        - comparison["Recall Mean_TSTR"]
    )

    comparison["Synthetic_Model"] = synth_name

    print(f"{synth_name} - TRTR vs TSTR")

    display(
        comparison[
            [
                "Synthetic_Model",
                "Model",
                "Accuracy_Drop",
                "F1_Drop",
                "Precision_Drop",
                "Recall_Drop",
                "Accuracy ± SD_TRTR",
                "Accuracy ± SD_TSTR"
            ]
        ]
    )

    all_comparisons.append(comparison)

combined_comparison = pd.concat(
    all_comparisons,
    ignore_index=True
)

summary = (
    combined_comparison
    .groupby(
        "Synthetic_Model",
        as_index=False
    )[
        [
            "Accuracy_Drop",
            "F1_Drop",
            "Precision_Drop",
            "Recall_Drop"
        ]
    ]
    .mean()
    .sort_values(
        "Accuracy_Drop",
        ascending=True
    )
)

print("Average metric drop by synthetic generator (lower is better)")

display(summary)


TRTR (Train Real, Test Real)


,Model,Accuracy ± SD,F1 ± SD,Precision ± SD,Recall ± SD
1,SVM-RBF,0.8890 ± 0.0030,0.8577 ± 0.0043,0.8680 ± 0.0073,0.8890 ± 0.0030
0,LogReg,0.8885 ± 0.0028,0.8570 ± 0.0045,0.8666 ± 0.0070,0.8885 ± 0.0028
7,GradientBoost,0.8884 ± 0.0036,0.8594 ± 0.0045,0.8657 ± 0.0082,0.8884 ± 0.0036
8,AdaBoost,0.8883 ± 0.0041,0.8584 ± 0.0056,0.8656 ± 0.0097,0.8883 ± 0.0041
9,MLP,0.8848 ± 0.0055,0.8586 ± 0.0069,0.8587 ± 0.0107,0.8848 ± 0.0055
5,RandomForest,0.8831 ± 0.0058,0.8575 ± 0.0066,0.8557 ± 0.0106,0.8831 ± 0.0058
2,KNN,0.8797 ± 0.0028,0.8544 ± 0.0038,0.8496 ± 0.0051,0.8797 ± 0.0028
6,ExtraTrees,0.8702 ± 0.0046,0.8487 ± 0.0047,0.8392 ± 0.0065,0.8702 ± 0.0046
3,NaiveBayes,0.8510 ± 0.0078,0.8419 ± 0.0068,0.8348 ± 0.0068,0.8510 ± 0.0078
4,DecisionTree,0.8092 ± 0.0098,0.8142 ± 0.0068,0.8197 ± 0.0046,0.8092 ± 0.0098


CTGAN - TSTR


,Model,Accuracy ± SD,F1 ± SD,Precision ± SD,Recall ± SD
1,SVM-RBF,0.8792 ± 0.0039,0.8358 ± 0.0058,0.8378 ± 0.0180,0.8792 ± 0.0039
0,LogReg,0.8782 ± 0.0034,0.8337 ± 0.0042,0.8334 ± 0.0152,0.8782 ± 0.0034
8,AdaBoost,0.8764 ± 0.0039,0.8403 ± 0.0056,0.8349 ± 0.0102,0.8764 ± 0.0039
7,GradientBoost,0.8734 ± 0.0064,0.8454 ± 0.0067,0.8372 ± 0.0110,0.8734 ± 0.0064
5,RandomForest,0.8354 ± 0.0071,0.8247 ± 0.0073,0.8156 ± 0.0081,0.8354 ± 0.0071
9,MLP,0.8331 ± 0.0156,0.8253 ± 0.0097,0.8191 ± 0.0069,0.8331 ± 0.0156
2,KNN,0.8200 ± 0.0106,0.8175 ± 0.0097,0.8152 ± 0.0095,0.8200 ± 0.0106
6,ExtraTrees,0.8120 ± 0.0100,0.8116 ± 0.0085,0.8113 ± 0.0074,0.8120 ± 0.0100
3,NaiveBayes,0.7481 ± 0.0201,0.7812 ± 0.0143,0.8326 ± 0.0049,0.7481 ± 0.0201
4,DecisionTree,0.7407 ± 0.0101,0.7702 ± 0.0075,0.8098 ± 0.0066,0.7407 ± 0.0101


CTGAN - TRTR vs TSTR


...

,Model,Accuracy ± SD,F1 ± SD,Precision ± SD,Recall ± SD
8,AdaBoost,0.8890 ± 0.0038,0.8585 ± 0.0057,0.8671 ± 0.0090,0.8890 ± 0.0038
0,LogReg,0.8880 ± 0.0014,0.8501 ± 0.0030,0.8712 ± 0.0046,0.8880 ± 0.0014
1,SVM-RBF,0.8877 ± 0.0019,0.8525 ± 0.0032,0.8669 ± 0.0059,0.8877 ± 0.0019
5,RandomForest,0.8843 ± 0.0048,0.8514 ± 0.0044,0.8573 ± 0.0123,0.8843 ± 0.0048
6,ExtraTrees,0.8837 ± 0.0019,0.8481 ± 0.0037,0.8541 ± 0.0056,0.8837 ± 0.0019
9,MLP,0.8813 ± 0.0049,0.8476 ± 0.0068,0.8485 ± 0.0124,0.8813 ± 0.0049
2,KNN,0.8772 ± 0.0019,0.8456 ± 0.0039,0.8402 ± 0.0053,0.8772 ± 0.0019
3,NaiveBayes,0.8589 ± 0.0076,0.8465 ± 0.0065,0.8381 ± 0.0070,0.8589 ± 0.0076
7,GradientBoost,0.8418 ± 0.0326,0.8300 ± 0.0155,0.8290 ± 0.0127,0.8418 ± 0.0326
4,DecisionTree,0.5886 ± 0.0613,0.6580 ± 0.0509,0.8034 ± 0.0099,0.5886 ± 0.0613


CTABGAN - TRTR vs TSTR


,Synthetic_Model,Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop,Accuracy ± SD_TRTR,Accuracy ± SD_TSTR
0,CTABGAN,SVM-RBF,0.00130,0.005151,0.001016,0.00130,0.8890 ± 0.0030,0.8877 ± 0.0019
1,CTABGAN,LogReg,0.00055,0.006925,-0.004634,0.00055,0.8885 ± 0.0028,0.8880 ± 0.0014
2,CTABGAN,GradientBoost,0.04660,0.029451,0.036686,0.04660,0.8884 ± 0.0036,0.8418 ± 0.0326
3,CTABGAN,AdaBoost,-0.00060,-0.000139,-0.001517,-0.00060,0.8883 ± 0.0041,0.8890 ± 0.0038
4,CTABGAN,MLP,0.00350,0.011000,0.010163,0.00350,0.8848 ± 0.0055,0.8813 ± 0.0049
5,CTABGAN,RandomForest,-0.00120,0.006049,-0.001648,-0.00120,0.8831 ± 0.0058,0.8843 ± 0.0048
6,CTABGAN,KNN,0.00245,0.008741,0.009354,0.00245,0.8797 ± 0.0028,0.8772 ± 0.0019
7,CTABGAN,ExtraTrees,-0.01345,0.000544,-0.014894,-0.01345,0.8702 ± 0.0046,0.8837 ± 0.0019
8,CTABGAN,NaiveBayes,-0.00790,-0.004569,-0.003257,-0.00790,0.8510 ± 0.0078,0.8589 ± 0.0076
9,CTABGAN,DecisionTree,0.22065,0.156187,0.016334,0.22065,0.8092 ± 0.0098,0.5886 ± 0.0613


Average metric drop by synthetic generator (lower is better)


,Synthetic_Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop
5,WGAN_GP,0.011010,0.034127,0.056670,0.011010
2,CopulaGAN,0.018595,0.008741,0.014222,0.018595
3,GaussianCopula,0.021010,0.034592,0.052738,0.021010
0,CTABGAN,0.025190,0.021934,0.004760,0.025190
4,TVAE,0.027940,0.028102,0.026457,0.027940
1,CTGAN,0.043595,0.032226,0.027637,0.043595


In [12]:
output_file = "TRTR_TSTR_results.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    trtr_results.to_excel(
        writer,
        sheet_name="TRTR_Results",
        index=False
    )

    combined_comparison.to_excel(
        writer,
        sheet_name="All_Comparisons",
        index=False
    )

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    for synth_name in model_order:
        synth_results = combined_comparison[
            combined_comparison["Synthetic_Model"] == synth_name
        ]

        synth_results.to_excel(
            writer,
            sheet_name=synth_name[:31],
            index=False
        )

print(f"Results saved to: {output_file}")

Results saved to: TRTR_TSTR_results.xlsx
